# Create Knowledge Probes
This generates our knowledge probes to test for factual recall.

In [1]:
import textwrap
import sys 
sys.path.append('../..')
import utils.utils as utils
import pandas as pd

with open('../../data/arxiv/cleaned_DPO.txt', 'r') as f:
    paper = f.read()


def print_wrapped(text, width=100):
    """
    Prints the given text wrapped to a specified width for better readability in notebooks.
    This function preserves paragraph breaks.
    """
    paragraphs = text.split('\n\n')
    for para in paragraphs:
        print(textwrap.fill(para, width=width))
        print()
        
print_wrapped(paper)

\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} While large-scale unsupervised language models (LMs) learn broad world knowledge
and some reasoning skills, achieving precise control of their behavior is difficult due to the
completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding optimal policy in clo

In [2]:
import re
import pandas as pd

def parse_paper_structure(text):
    """Parse paper into sections, subsections and paragraphs with metadata."""
    sections = []
    
    # Split by sections first
    section_pattern = r'\\section\{([^}]+)\}'
    section_splits = re.split(section_pattern, text)
    
    current_section = "Title/Abstract"
    current_section_content = ""
    
    for i in range(len(section_splits)):
        if i == 0:
            # Content before first section
            content = section_splits[i]
            current_section_content = content
        elif i % 2 == 1:
            # This is a section title
            current_section = section_splits[i]
            continue
        else:
            # This is section content
            content = section_splits[i]
            current_section_content = content
        
        # Now split by subsections within this section
        subsection_pattern = r'\\subsection\{([^}]+)\}'
        subsection_splits = re.split(subsection_pattern, content)
        
        current_subsection = "No Subsection"
        current_subsection_content = ""
        
        for j in range(len(subsection_splits)):
            if j == 0:
                # Content before first subsection
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            elif j % 2 == 1:
                # This is a subsection title
                current_subsection = subsection_splits[j]
                continue
            else:
                # This is subsection content
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            
            # Split into paragraphs
            paragraphs = [p.strip() for p in subsection_content.split('\n\n') if p.strip()]
            
            for paragraph in paragraphs:
                sections.append({
                    'section': current_section,
                    'subsection': current_subsection,
                    'paragraph': paragraph,
                    'section_text': current_section_content,
                    'subsection_text': current_subsection_content
                })
    
    return pd.DataFrame(sections)

### 1. Extracting Knowledge

This is to tag sentences that contain information, knowledge, and facts. From these we will extract self-contained, atomic facts.

In [15]:
extraction_prompt = r"""Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all sentences that contain "pieces of knowledge" or "facts."

# What to Exclude (Do NOT tag these):
- Sentences that are transitional and for structural purposes of the paper, mainly containing language that's generic to any paper, adding zero information e.g. "Our results raise several important questions for future work.", "In this section, we discuss our methodology in relation to other works.
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Latex commands that generate figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all sentences that contain a "piece of knowledge" or a "fact" and do not fall into the exclusion categories.
3.  Wrap each of these sentences in `<knowledge>` and `</knowledge>` tags.
4.  You can tag *captions* of the table or figure, but please DO NOT tag the other parts of the table/figure.
5.  For sentences that contain latex commands, place the tags so that it includes any latex code that's part of the sentence e.g. "\begin{definition}" or "\text{...}".
6.  For sentences that contain math, make sure to include all the latex of the math within the tags.
7.  Please make sure the tags cover the ENTIRE sentence i.e. the tags are at the beginning and end of the sentence.
8.  Return the entire original text with these annotations. Do not modify or summarize the text itself."""

# Parse paper structure
paper_df = parse_paper_structure(paper)

# Process each paragraph with LLM
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""{paragraph}"""
    return utils.query_llm(prompt, model='gpt-4.1')

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, row['paragraph']) for _, row in paper_df.iterrows()]
    extracted_claims = [future.result() for future in futures]

# Add extracted claims to dataframe
paper_df['extracted_claims'] = extracted_claims

# Print each output with section/subsection context
for i, (_, row) in enumerate(paper_df.iterrows(), 1):
    print_wrapped(f"Section: {row['section']}")
    print_wrapped(f"Subsection: {row['subsection']}")
    print_wrapped(f"Paragraph {i}: {row['extracted_claims']}")
    print_wrapped("-" * 50)

Section: Title/Abstract

Subsection: No Subsection

Paragraph 1: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

--------------------------------------------------

Section: Title/Abstract

Subsection: No Subsection

Paragraph 2: \begin{abstract} While large-scale unsupervised language models (LMs) learn broad world
knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to
the completely unsupervised nature of their training. <knowledge>Existing methods for gaining such
steerability collect human labels of the relative quality of model generations and fine-tune the
unsupervised LM to align with these preferences, often with reinforcement learning from human
feedback (RLHF).</knowledge> <knowledge>However, RLHF is a complex and often unstable procedure,
first fitting a reward model that reflects the human preferences, and then fine-tuning the large
unsupervised LM using reinforcement learning to maximize 

#### 1.1 Check Knowledge is actually in the paper

In [17]:
import re
import pandas as pd

def extract_text_from_knowledge_tags(text: str) -> list[str]:
    """
    Finds all <knowledge> tags in a given text and extracts their content.

    Args:
        text: A string containing the text to parse, which may include
              <knowledge>...</knowledge> tags.

    Returns:
        A list of strings, where each string is the content found within
        a <knowledge> tag. The content is stripped of leading/trailing
        whitespace.
    """
    # This regex pattern finds all content between <knowledge> and </knowledge>.
    # - The (.*?) part is a non-greedy capture group for the content inside the tags.
    # - The re.DOTALL flag allows the '.' character to match newlines, so tags
    #   that span multiple lines are correctly handled.
    pattern = re.compile(r'<knowledge>(.*?)</knowledge>', re.DOTALL)
    
    # re.findall returns a list of all captured groups.
    matches = pattern.findall(text)
    
    # Clean up any leading/trailing whitespace from the extracted text.
    cleaned_matches = [match.strip() for match in matches]
    
    return cleaned_matches

def remove_knowledge_tags(text: str) -> str:
    """Remove knowledge tags from text while preserving the content."""
    pattern = re.compile(r'</?knowledge>', re.DOTALL)
    return pattern.sub('', text)

paper_df['extracted_claims'] = extracted_claims

# Extract a list of knowledge statements for each row
paper_df['knowledge_list'] = paper_df['extracted_claims'].apply(extract_text_from_knowledge_tags)

# Explode the DataFrame on the knowledge_list column
paper_df_exploded = paper_df.explode('knowledge_list').rename(columns={'knowledge_list': 'raw_knowledge_statement'})

# Drop rows with no knowledge statements
paper_df_exploded = paper_df_exploded[paper_df_exploded['raw_knowledge_statement'].notna()]

# Count total claims extracted by the LLM before filtering
total_extracted_claims = paper_df_exploded['raw_knowledge_statement'].notna().sum()

# Filter out claims that are not actually in the original paper text (case-insensitive)
paper_lower = paper.lower()
def is_claim_in_paper(claim):
    # Rows with no knowledge statement (claim is NaN) are kept
    if pd.isna(claim):
        return True
    # Check if the lowercased claim is in the lowercased paper
    return claim.strip().lower() in paper_lower

# Apply the filter and create a new validated dataframe
paper_df_validated = paper_df_exploded[paper_df_exploded['raw_knowledge_statement'].apply(is_claim_in_paper)].copy()

# Count claims that passed validation
validated_claims_count = paper_df_validated['raw_knowledge_statement'].notna().sum()
print(f"Found {validated_claims_count}/{total_extracted_claims} extracted claims in the original paper text.")

# Clean up the original paragraph text by removing knowledge tags from the validated dataframe
paper_df_validated['paragraph'] = paper_df_validated['extracted_claims'].apply(remove_knowledge_tags)

# Drop the now-redundant columns
paper_df_validated = paper_df_validated.drop(columns=['extracted_claims'])

# Display the result
print(f"Total rows after exploding and validation: {len(paper_df_validated)}")
paper_df_validated

Found 197/197 extracted claims in the original paper text.
Total rows after exploding and validation: 197


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ..."
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit..."
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Our experiments show that DPO can fine-tune LM...
...,...,...,...,...,...,...
39,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Rather than coercing the preference learning p...
39,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","With virtually no tuning of hyperparameters, D..."
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Our initial results suggest that DPO policies ...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra..."


In [5]:
# Print all the NAs
na_rows = paper_df_validated[paper_df_validated['raw_knowledge_statement'].isna()]
print(f"Found {len(na_rows)} rows with NA knowledge statements:")
for idx, row in na_rows.iterrows():
    print(f"\nRow {idx}:")
    print(f"Paragraph: {row['paragraph']}")

Found 0 rows with NA knowledge statements:


### 2. Filter Bad Sentences

Not all extracted knowledge statements may be valid. We double check that the extracted statements meet our requirements for a proper probe.

##### 2.1 Filter Out Predominantly LaTeX Statements
This is to avoid figures, tables, and sentences that are dominated by LaTeX that prevents any suitable English target. LaTeX has various valid formats which can make evaluation tricky. There's also the chance that LaTeX introduces noise with regards to learnability. While OLMo has been trained on arxiv documents that include latex,it potentially may require further pre-training on mathematical notation and latex for the LLM to understand these statements.

In [18]:
# Filter out knowledge statements that are more than 50% LaTeX
def calculate_latex_percentage(text):
    """
    Calculate the percentage of LaTeX/mathematical content in a text string.
    
    Args:
        text (str): The text to analyze
        
    Returns:
        float: Percentage of text that is LaTeX/mathematical (0-100)
    """
    if pd.isna(text) or not text.strip():
        return 0.0
    
    import re
    
    total_chars = len(text)
    latex_chars = 0
    
    # Count LaTeX commands (backslash followed by letters)
    latex_commands = re.findall(r'\\[a-zA-Z]+', text)
    for cmd in latex_commands:
        latex_chars += len(cmd)
    
    # Remove LaTeX commands to avoid double counting
    text_without_commands = re.sub(r'\\[a-zA-Z]+', '', text)
    
    # Count non-alphabetic characters in the remaining text
    for char in text_without_commands:
        if not char.isalpha() and not char.isspace():
            latex_chars += 1
    
    # Calculate percentage
    latex_percentage = (latex_chars / total_chars) * 100 if total_chars > 0 else 0.0
    
    return latex_percentage

# Apply the filter
paper_df_validated['latex_percentage'] = paper_df_validated['raw_knowledge_statement'].apply(calculate_latex_percentage)

# Filter out statements with more than 50% LaTeX
latex_threshold = 50
paper_df_filtered = paper_df_validated[paper_df_validated['latex_percentage'] <= latex_threshold].copy()

# Report filtering results
total_before = len(paper_df_validated)
total_after = len(paper_df_filtered)
filtered_out = total_before - total_after

print(f"LaTeX filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} statements with >{latex_threshold}% LaTeX content")

# Show all filtered statements
if filtered_out > 0:
    high_latex_statements = paper_df_validated[paper_df_validated['latex_percentage'] > latex_threshold]
    print(f"\nAll {filtered_out} filtered statements (high LaTeX content):")
    for idx, row in high_latex_statements.iterrows():
        print(f"  Row {idx}: LaTeX {row['latex_percentage']:.1f}%")
        print(f"    Statement: '{row['raw_knowledge_statement']}'")
        print()
        
paper_df_filtered.reset_index(drop=True, inplace=True)


LaTeX filtering results:
  Before filtering: 197 knowledge statements
  After filtering: 191 knowledge statements
  Filtered out: 6 statements with >50% LaTeX content

All 6 filtered statements (high LaTeX content):
  Row 16: LaTeX 58.3%
    Statement: 'where $Z(x) =\sum_{y}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)$ is the partition function.'

  Row 16: LaTeX 53.3%
    Statement: 'Thus, the optimal RLHF policy $\pi^*$ under the Bradley-Terry model satisfies the preference model:
\begin{equation}\label{eq:objective}
    p^*(y_1\succ y_2 \mid x)=\frac{1}{1 + \exp\left(\beta \log \frac{\pi^*(y_2\mid x)}{\piref(y_2\mid x)} - \beta \log \frac{\pi^*(y_1\mid x)}{\piref(y_1\mid x)}\right)}
\end{equation}'

  Row 17: LaTeX 60.0%
    Statement: 'Analogous to the reward modeling approach (i.e. Eq.~\ref{eq:reward_model}), our policy objective becomes:
\begin{equation}\label{eq:optimum_model}
    \mathcal{L}_\text{DPO}(\pi_{\theta}; \piref) = -\mathbb{E}_{(x, y_w, y_l)\sim \mathcal{D}

##### 2.2 Filter Out Sentences with References Not In Context

Some sentences contain references that are defined in a section of the paper that does not jointly appear during training since it's in a different section. Technically it may have appeared together if it's a small subsection since we join subsections that are small together, but they are at the very least far away. We filter out sentences that have such references.


In [19]:
import re
import pandas as pd

def find_undefined_references(sentence: str, subsection_text: str) -> list[str]:
    """
    Finds LaTeX references in a sentence that are not defined in the given subsection text.

    This function identifies all references formatted as \\ref{...} in the input sentence.
    It then checks for corresponding \\label{...} definitions within the subsection_text.
    
    Args:
        sentence (str): The sentence to check for references.
        subsection_text (str): The text of the subsection to check for labels.

    Returns:
        list[str]: A list of reference labels that are used in the sentence but not
                   defined in the subsection text. An empty list indicates all
                   references are defined locally.
    """
    # Find all references in the sentence, e.g., \ref{eq:RL} -> "eq:RL"
    references = re.findall(r'\\ref\{([^}]+)\}', sentence)
    if not references:
        return True

    # Find all defined labels in the subsection text, e.g., \label{eq:main_eq} -> "eq:main_eq"
    defined_labels = set(re.findall(r'\\label\{([^}]+)\}', subsection_text))

    # Identify references that are not defined within the subsection
    undefined_references = [ref for ref in references if ref not in defined_labels]

    if len(undefined_references) > 0:
        return False
    else:
        return True

keep = paper_df_filtered.apply(
    lambda row: find_undefined_references(row['raw_knowledge_statement'], row['subsection_text']),
    axis=1
)

# Report filtering results
total_before = len(paper_df_filtered)
dropped_statements = paper_df_filtered[~keep]
total_after = sum(keep)
filtered_out = total_before - total_after

print(f"Reference filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} statements with undefined references")

# Show all dropped statements
if filtered_out > 0:
    print(f"\nAll {filtered_out} dropped statements (undefined references):")
    for idx, row in dropped_statements.iterrows():
        print(f"  Row {idx}: '{row['raw_knowledge_statement']}'")
        print()

paper_df_filtered = paper_df_filtered[keep].reset_index(drop=True)


Reference filtering results:
  Before filtering: 191 knowledge statements
  After filtering: 166 knowledge statements
  Filtered out: 25 statements with undefined references

All 25 dropped statements (undefined references):
  Row 63: 'We start with the same RL objective as prior work, Eq.~\ref{eq:RL}, under a general reward function $r$.'

  Row 64: 'Following prior work~\citep{peters2007reinforcement, peng2019advantage, korbak2022reinforcement, go2023aligning}, it is straightforward to show that the optimal solution to the KL-constrained reward maximization objective in Eq.~\ref{eq:RL} takes the form:
\begin{equation}\label{eq:op_policy}
    \pi_r(y\mid x) = \frac{1}{Z(x)}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right),
\end{equation}%'

  Row 65: 'See Appendix \ref{app:derivation1} for a complete derivation.'

  Row 71: 'Substituting the reparameterization in Eq.~\ref{eq:main_eq} for $r^*(x,y)$ into the preference model Eq.~\ref{eq:bradley-terry}, the partition function cance

##### 2.3 Filter Short Facts

In [20]:
# Filter out knowledge statements that are too short
min_length = 90
keep_length = paper_df_filtered['raw_knowledge_statement'].str.len() >= min_length

# Report filtering results
total_before = len(paper_df_filtered)
dropped_statements = paper_df_filtered[~keep_length]
total_after = sum(keep_length)
filtered_out = total_before - total_after

print(f"Length filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} statements shorter than {min_length} characters")

# Show all dropped statements
if filtered_out > 0:
    print(f"\nAll {filtered_out} dropped statements (too short):")
    for idx, row in dropped_statements.iterrows():
        print(f"  Row {idx}: '{row['raw_knowledge_statement']}' (length: {len(row['raw_knowledge_statement'])})")

paper_df_filtered = paper_df_filtered[keep_length].reset_index(drop=True)


Length filtering results:
  Before filtering: 166 knowledge statements
  After filtering: 158 knowledge statements
  Filtered out: 8 statements shorter than 90 characters

All 8 dropped statements (too short):
  Row 54: 'In practice, the language model policy $\pi_\theta$ is also initialized to $\pisft$.' (length: 84)
  Row 99: '\textbf{Tasks.} Our experiments explore three different open-ended text generation tasks.' (length: 89)
  Row 112: 'Our experiments use two different approaches to evaluation.' (length: 59)
  Row 128: 'This sweep includes 22 runs in total.' (length: 37)
  Row 131: 'This result is particularly notable for multiple reasons.' (length: 57)
  Row 139: 'DPO also achieves a higher maximum win rate compared to the best of $N$ baseline.' (length: 81)
  Row 142: 'Preferred-FT does not improve significantly over the SFT model.' (length: 63)
  Row 150: 'The results are presented in Table~\ref{tab:ood}.' (length: 49)


##### 2.4 Filter Unsuitable Probes

Sometimes the sentences that are extracted are trivial sentences that simply exist for transitonal purposes, or are rhetorical questions, etc. and essentially shouldn't have been tagged the first time. This was specified on the 1st extraction attempt, but we ensure that this is the case. We also check for extraction that failed and led to corrupted sentences i.e. cutoff in the middle. We specify the types of sentences that are *unsuitable* and make sure to filter out these sentences.

In [9]:
from tqdm import tqdm
from importlib import reload
reload(utils)
import json
# Evaluate knowledge statements for suitability
prompt = {}
prompt['system'] = """You are a meticulous, sharp, and detail-oriented evaluator.

# Instructions
You will be receiving clauses from an academic paper. Your task is to determine whether the clause is suitable for testing an LLM's factual recall. You will go about checking each of the following criteria for exclusion. If it satisfies any of these criteria, it is unsuitable. Note, one of the conditions ask if the sentence contains a a valid *target*. A target of a clause is a phrase, 1-2 words long, that is among the key, central information in that clause.

The clause is UNSUITABLE if it:
1. Mostly consists of mathematical expressions such that the only valid targets in the clause are mathematical expressions in LaTeX. 
 a. Plain english embedded within LaTeX commands, such as captions or italics, is SUITABLE and should NOT be considered unsuitable.
 b. If at least one valid English target can be found, it should NOT be considered unsuitable.
2. The clause only redirects to a section of the paper without adding any additional information (e.g., "This is discussed in Section 3.1"). If the clause also contains other information, it should NOT be considered unsuitable.
3. The clause is a rhetorical question.
4. The clause is not valid: it's unclear, cutoff in the middle, or shows displays of corruption.

# Output Format
Respond with JSON format with the following keys:
- "suitable": boolean (true/false)
- "unsuitable_condition": integer (1, 2, 3, or 4) or null if suitable
- "explanation": string"""

# Apply evaluation to the first knowledge statement only
print("Evaluating knowledge statement suitability...")

row = paper_df_filtered.iloc[93]
statement = row['raw_knowledge_statement']
user_prompt = f"Clause: {statement}"
print(user_prompt)
full_prompt = {
    'system': prompt['system'],
    'user': user_prompt
}

result = utils.query_llm(full_prompt, model='gpt-4.1', return_json=True, max_tokens=1000)
result = json.loads(result)
is_suitable = result['suitable']
unsuitable_condition = result.get('unsuitable_condition')
explanation = result.get('explanation')
print(f"First statement result:")
print(f"  Suitable: {is_suitable}")
print(f"  Unsuitable condition: {unsuitable_condition}")
print(f"  Explanation: {explanation}")


Evaluating knowledge statement suitability...
Clause: In this setting, we can interpret the normalization term in $f(r_{\phi}, \piref, \beta)$ as the soft value function of the reference policy $\piref$.
First statement result:
  Suitable: True
  Unsuitable condition: None
  Explanation: The clause contains a valid English target ('soft value function', 'reference policy') and is not just a mathematical expression, a section redirect, a rhetorical question, or an invalid/corrupted sentence.


In [21]:
from tqdm import tqdm
from importlib import reload
reload(utils)
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Evaluate knowledge statements for suitability
prompt = {}
prompt['system'] = """You are a meticulous, sharp, and detail-oriented evaluator.

# Instructions
You will be receiving clauses from an academic paper. Your task is to determine whether the clause is suitable for testing an LLM's factual recall. You will go about checking each of the following criteria for exclusion. If it satisfies any of these criteria, it is unsuitable. Note, one of the conditions ask if the sentence contains a a valid *target*. A target of a clause is a phrase, 1-2 words long, that is among the key, central information in that clause.

The clause is UNSUITABLE if it:
1. Mostly consists of mathematical expressions such that the only valid targets in the clause are mathematical expressions in LaTeX. 
 a. Plain english embedded within LaTeX commands, such as captions or italics, is SUITABLE and should NOT be considered unsuitable.
 b. If at least one valid English target can be found, it should NOT be considered unsuitable.
2. The clause provides meta-commentary of the paper or points to a section of the paper without adding additional information. 
    a. For example, "Our results raise several important questions for future work." or "In this section, we discuss our methodology." or "See Section 3.1."
    b. If the clause contains more than generic language and actually contains details about the paper, it should NOT be considered unsuitable.
3. The clause is a rhetorical question.
4. The clause is a caption that describes what the table contains but doesn't actually discuss the contents i.e. doesn't actually add any information.
5. The clause is confusing: it's unclear, grammatically confusing, or cutoff in the middle.

# Output Format
Respond with JSON format with the following keys:
- "unsuitable_condition": integer (1, 2, 3, 4, or 5) or null if suitable
- "suitable": boolean (true/false)"""

def evaluate_statement(idx_row):
    idx, row = idx_row
    statement = row['raw_knowledge_statement']
    user_prompt = f"Context: {row['paragraph']}\n\nClause: {statement}"
    full_prompt = {
        'system': prompt['system'],
        'user': user_prompt
    }
    
    result = utils.query_llm(full_prompt, model='o4-mini', return_json=True, max_tokens=50)
    result = json.loads(result)
    is_suitable = result['suitable']
    unsuitable_condition = result.get('unsuitable_condition')
    
    return idx, is_suitable, unsuitable_condition

# Run evaluation 3 times and store separate results
print("Evaluating knowledge statement suitability (3 rounds)...")

# Print first statement
first_statement = paper_df_filtered.iloc[0]['raw_knowledge_statement']
print(f"Clause: {first_statement}")

# Initialize results for 3 rounds
suitability_results_1 = [None] * len(paper_df_filtered)
suitability_results_2 = [None] * len(paper_df_filtered)
suitability_results_3 = [None] * len(paper_df_filtered)
unsuitable_conditions_1 = [None] * len(paper_df_filtered)
unsuitable_conditions_2 = [None] * len(paper_df_filtered)
unsuitable_conditions_3 = [None] * len(paper_df_filtered)

for round_num in range(3):
    print(f"\nRound {round_num + 1}/3...")
    
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(evaluate_statement, (idx, row)): idx 
                   for idx, row in paper_df_filtered.iterrows()}
        
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Round {round_num + 1}"):
            idx, is_suitable, unsuitable_condition = future.result()
            original_idx = list(paper_df_filtered.index).index(idx)
            
            if round_num == 0:
                suitability_results_1[original_idx] = is_suitable
                unsuitable_conditions_1[original_idx] = unsuitable_condition
            elif round_num == 1:
                suitability_results_2[original_idx] = is_suitable
                unsuitable_conditions_2[original_idx] = unsuitable_condition
            else:
                suitability_results_3[original_idx] = is_suitable
                unsuitable_conditions_3[original_idx] = unsuitable_condition
        print(f"Round {round_num + 1} done...")

# Add all suitability columns
paper_df_filtered['is_suitable_1'] = suitability_results_1
paper_df_filtered['is_suitable_2'] = suitability_results_2
paper_df_filtered['is_suitable_3'] = suitability_results_3
paper_df_filtered['unsuitable_condition_1'] = unsuitable_conditions_1
paper_df_filtered['unsuitable_condition_2'] = unsuitable_conditions_2
paper_df_filtered['unsuitable_condition_3'] = unsuitable_conditions_3

# Count how many rounds marked each statement as unsuitable
paper_df_filtered['unsuitable_count'] = (~paper_df_filtered['is_suitable_1']).astype(int) + \
                                        (~paper_df_filtered['is_suitable_2']).astype(int) + \
                                        (~paper_df_filtered['is_suitable_3']).astype(int)

# Create final suitability column (unsuitable if at least 2 out of 3 rounds marked it as unsuitable)
paper_df_filtered['is_suitable'] = paper_df_filtered['unsuitable_count'] < 2

paper_df_suitable = paper_df_filtered[paper_df_filtered['is_suitable']].copy()

# Report filtering results
total_before = len(paper_df_filtered)
total_after = len(paper_df_suitable)
filtered_out = total_before - total_after

print(f"\nFinal suitability filtering results (at least 2 out of 3 rounds):")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} unsuitable statements")

Evaluating knowledge statement suitability (3 rounds)...
Clause: Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).

Round 1/3...


Round 1: 100%|██████████| 158/158 [00:39<00:00,  4.02it/s]


Round 1 done...

Round 2/3...


Round 2: 100%|██████████| 158/158 [00:39<00:00,  3.95it/s]


Round 2 done...

Round 3/3...


Round 3: 100%|██████████| 158/158 [00:43<00:00,  3.66it/s]

Round 3 done...

Final suitability filtering results (at least 2 out of 3 rounds):
  Before filtering: 158 knowledge statements
  After filtering: 150 knowledge statements
  Filtered out: 8 unsuitable statements


In [23]:


# Show statements that got only one unsuitable
one_unsuitable = paper_df_filtered[paper_df_filtered['unsuitable_count'] == 1]
if len(one_unsuitable) > 0:
    print(f"\nStatements that got only one unsuitable (still kept):")
    for idx, row in one_unsuitable.iterrows():
        conditions = []
        if pd.notna(row['unsuitable_condition_1']) and not row['is_suitable_1']:
            conditions.append(f"Round 1: {row['unsuitable_condition_1']}")
        if pd.notna(row['unsuitable_condition_2']) and not row['is_suitable_2']:
            conditions.append(f"Round 2: {row['unsuitable_condition_2']}")
        if pd.notna(row['unsuitable_condition_3']) and not row['is_suitable_3']:
            conditions.append(f"Round 3: {row['unsuitable_condition_3']}")
        condition_str = ", ".join(conditions) if conditions else "None"
        print_wrapped(f"  Row {idx} ({condition_str}): '{row['raw_knowledge_statement']}'")


Statements that got only one unsuitable (still kept):
  Row 4 (Round 3: 2.0): 'Our experiments show that DPO can fine-tune LMs to align with human
preferences as well as or better than existing methods.'

  Row 24 (Round 2: 2.0): 'Our main contribution is Direct Preference Optimization (DPO), a simple
RL-free algorithm for training language models from preferences.'

  Row 40 (Round 2: 2.0): 'We instead present a single stage policy learning approach that directly
optimizes a policy to satisfy preferences.'

  Row 53 (Round 3: 1.0): 'Following prior works~\citep{jaques2017sequence, jaques2020human}, the
optimization is formulated as \begin{equation}\label{eq:RL} \max_{\pi_{\theta}}  \mathbb{E}_{x\sim
\mathcal{D}, y\sim \pi_{\theta}(y \mid x)}\bigl[r_{\phi}(x, y)\bigr] -
\beta\mathbb{D}_{\textrm{KL}}\bigl[\pi_{\theta}(y\mid x)\mid \mid \piref(y\mid x)\bigr],
\end{equation} where $\beta$ is a parameter controlling the deviation from the base reference policy
$\piref$, namely the initial

In [22]:
# Show all unsuitable statements with their conditions
if filtered_out > 0:
    unsuitable_statements = paper_df_filtered[~paper_df_filtered['is_suitable']]
    print(f"\nAll {filtered_out} unsuitable statements:")
    for idx, row in unsuitable_statements.iterrows():
        conditions = []
        if pd.notna(row['unsuitable_condition_1']):
            conditions.append(str(row['unsuitable_condition_1']))
        if pd.notna(row['unsuitable_condition_2']):
            conditions.append(str(row['unsuitable_condition_2']))
        if pd.notna(row['unsuitable_condition_3']):
            conditions.append(str(row['unsuitable_condition_3']))
        condition_str = ", ".join(conditions) if conditions else "None"
        print_wrapped(f"  Row {idx} (Conditions {condition_str}): '{row['raw_knowledge_statement']}'")


All 8 unsuitable statements:
  Row 18 (Conditions 2.0, 2.0, 2.0): 'In this paper, we show how to directly optimize a language
model to adhere to human preferences, without explicit reward modeling or reinforcement learning.'

  Row 33 (Conditions 2.0, 2.0): 'Despite the appeal of using relative human preferences, fine-tuning
large language models with reinforcement learning remains a major practical challenge; this work
provides a theoretically-justified approach to optimizing relative preferences without RL.'

  Row 64 (Conditions 1.0, 1.0): 'Specifically, we first take the logarithm of both sides of
Eq.~\ref{eq:op_policy} and then with some algebra we obtain: \begin{equation}\label{eq:main_eq}
r(x,y) =\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)} + \beta \log Z(x). \end{equation}'

  Row 78 (Conditions 2.0, 2.0, 2.0): 'In this section we will build the theory behind this
reparameterization, show that it does not constrain the class of learned reward models, and allows
for the ex

In [24]:
# Show all suitable statements
if total_after > 0:
    print(f"\nAll {total_after} suitable statements:")
    for idx, row in paper_df_suitable.iterrows():
        print_wrapped(f"  Row {idx}: '{row['raw_knowledge_statement']}'")


All 150 suitable statements:
  Row 0: 'Existing methods for gaining such steerability collect human labels of the relative
quality of model generations and fine-tune the unsupervised LM to align with these preferences,
often with reinforcement learning from human feedback (RLHF).'

  Row 1: 'However, RLHF is a complex and often unstable procedure, first fitting a reward model that
reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement
learning to maximize this estimated reward without drifting too far from the original model.'

  Row 2: 'In this paper we introduce a new parameterization of the reward model in RLHF that enables
extraction of the corresponding optimal policy in closed form, allowing us to solve the standard
RLHF problem with only a simple classification loss.'

  Row 3: 'The resulting algorithm, which we call \textit{Direct Preference Optimization} (DPO), is
stable, performant, and computationally lightweight, eliminating the 

##### 2.5 Extract Title

In [25]:
import re
paper_df_suitable.reset_index(drop=True, inplace=True)
paper_df_suitable['title'] = re.search(r'\\title{(.*?)}', paper).group(1) if re.search(r'\\title{(.*?)}', paper) else None

### 3. Extract Self-Contained, Atomic Probes
Given the original, source sentences from the paper, we break each sentence into the parts that presents a new fact.


In [ ]:
import concurrent.futures
# You will be given a piece of text. The text is written such that each sentence is contextualized, which is not desirable for breaking the text apart into stand alone facts. Your task is to extract and rewrite the information into self-contained, atomic facts and that can be separated into a probe and a target. 

# First, segment the text into coherent subtexts. For each subtext, extract at most three *self-contained, atomic facts*, each a single declarative claim that can stand alone. Rewrite every fact so that it ends with a 1–3 word *target* capturing the key information. The *probe* is the same sentence with the *target* removed; it should read naturally and implicitly ask for the missing key information. 

# - Prefer precise, contentful targets (e.g., “reasoning path”, “human feedback”) over generic terms; keep targets to 1–3 words only. 
# - Do not over-extract: include only facts warranted by the text and extract at most 3 facts per subtext.
# - Preserve the original meaning and scope, avoid introducing new claims, and ensure the last words of each sentence are exactly the target. 
# - Present your results in the demonstrated Probe/Target format.
#I am trying to create self-contained, atomic knowledge probes for a language model, to measure its ability to recall facts. 
def extract_atomic_facts(sentence, paragraph):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For more complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. Often times, there are multiple valid targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

# Demonstrations
The next few demonstrations will be based on the same paragraph. 

Paragraph: "Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.
\begin{enumerate}[topsep=1pt,itemsep=0ex]%
    \item First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.
    \item Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question).
    \item Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.
    \item Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.
\end{enumerate}"

### Example 1
Sentence: "First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps." 

Targets:
- "intermediate steps"
- "reasoning steps"

Output:
- Probe: "Chain of thought allows models to decompose multi-step problems into", Target: "intermediate steps"
- Probe: "By decomposing multi-step problems into intermediate steps, chain of thought allows additional computation to be allocated to problems that require more", Target: "reasoning steps"

### Example 2
Sentence: "Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question)."

Targets:
- "interpretable window"
- "reasoning path"

Output:
- Probe: "Chain-of-thought shows how the model formed an answer and provides opportunities to debug mistakes in the", Target: "reasoning path"
- Probe: "The behavior of language models can be difficult to characterize but chain-of-thought provides an", Target: "interpretable window"

### Example 3
Sentence: "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting."

Targets:
- "including examples"
- "elicited"

Output:
- Probe: "Chain-of-thought reasoning can be elicited in sufficiently large off-the-shelf language models simply by", Target: "including examples"
- Probe: "By including examples of chain of thought sequences, chain-of-thought reasoning in sufficiently large off-the-shelf language models can be readily", Target: "elicited"

The next few examples will be based on the following paragraph for context. 
Paragraph: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

### Example 4
Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "behavior"
- "unsupervised training"

Output:
- Probe: "Large-scale unsupervised LMs learn broad", Target: "world knowledge" 
- Probe: "Large-scale unsupervised LMs learn", Target: "reasoning skills" 
- Probe: NA for "precise control" (Difficult to place at end of sentence)
- Probe: "Due to the completely unsupervised nature of the training of large-scale unsupervised language models (LMs), it is difficult to achieve precise control of their", Target: "behavior"
- Probe: "Achieving precise control of the behavior of unsupervised language models is difficult due to their", Target: "unsupervised training"

### Example 5
Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Output:
- Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for gaining such steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "Existing methods align unsupervised language models by fine-tuning on human", Target: "preferences"
"""
    prompt['user'] = f"""### Input\nContext: {paragraph}\n\nSentence: {sentence}### Output\n"""
    if len(paragraph.strip()) < 50:
        return None
    return utils.query_llm(prompt, model='gpt-5')

# Process first row only for testing
first_row = paper_df_suitable.iloc[2]
atomic_facts = extract_atomic_facts(first_row['raw_knowledge_statement'], first_row['paragraph'])

print(f"Extracted atomic facts for first row: {atomic_facts}")

Extracted atomic facts for first row: - Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for increasing steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "To align the behavior of unsupervised language models with desired outcomes, existing methods fine-tune on human", Target: "preferences"
- Probe: "Fine-tuning to align unsupervised language models with human preferences is often performed using reinforcement learning from human feedback, abbreviated as", Target: "RLHF"


In [ ]:
import concurrent.futures
# You will be given a piece of text. The text is written such that each sentence is contextualized, which is not desirable for breaking the text apart into stand alone facts. Your task is to extract and rewrite the information into self-contained, atomic facts and that can be separated into a probe and a target. 

# First, segment the text into coherent subtexts. For each subtext, extract at most three *self-contained, atomic facts*, each a single declarative claim that can stand alone. Rewrite every fact so that it ends with a 1–3 word *target* capturing the key information. The *probe* is the same sentence with the *target* removed; it should read naturally and implicitly ask for the missing key information. 

# - Prefer precise, contentful targets (e.g., “reasoning path”, “human feedback”) over generic terms; keep targets to 1–3 words only. 
# - Do not over-extract: include only facts warranted by the text and extract at most 3 facts per subtext.
# - Preserve the original meaning and scope, avoid introducing new claims, and ensure the last words of each sentence are exactly the target. 
# - Present your results in the demonstrated Probe/Target format.
#I am trying to create self-contained, atomic knowledge probes for a language model, to measure its ability to recall facts. 
def extract_atomic_facts(sentence, paragraph):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For more complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. Often times, there are multiple valid targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

# Demonstrations
The next few demonstrations will be based on the same paragraph. 

Paragraph: "Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.
\begin{enumerate}[topsep=1pt,itemsep=0ex]%
    \item First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.
    \item Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question).
    \item Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.
    \item Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.
\end{enumerate}"

### Example 1
Sentence: "First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps." 

Targets:
- "intermediate steps"
- "reasoning steps"

Contextualized Probes:
- Probe: "Chain of thought allows models to decompose multi-step problems into", Target: "intermediate steps"
- Probe: "By decomposing multi-step problems into intermediate steps, chain of thought allows additional computation to be allocated to problems that require more", Target: "reasoning steps"

### Example 2
Sentence: "Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question)."

Targets:
- "interpretable window"
- "reasoning path"

Contextualized Probes:
- Probe: "Chain-of-thought shows how the model formed an answer and provides opportunities to debug mistakes in the", Target: "reasoning path"
- Probe: "The behavior of language models can be difficult to characterize but chain-of-thought provides an", Target: "interpretable window"

### Example 3
Sentence: "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting."

Targets:
- "including examples"
- "elicited"

Contextualized Probes:
- Probe: "Chain-of-thought reasoning can be elicited in sufficiently large off-the-shelf language models simply by", Target: "including examples"
- Probe: "By including examples of chain of thought sequences, chain-of-thought reasoning in sufficiently large off-the-shelf language models can be readily", Target: "elicited"

The next few examples will be based on the following paragraph for context. 
Paragraph: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

### Example 4
Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "behavior"
- "unsupervised training"

Contextualized Probes:
- Probe: "Large-scale unsupervised LMs learn broad", Target: "world knowledge" 
- Probe: "Large-scale unsupervised LMs learn", Target: "reasoning skills" 
- Probe: NA (Difficult to place "precise control" at end of sentence)
- Probe: "Due to the completely unsupervised nature of the training of large-scale unsupervised language models (LMs), it is difficult to achieve precise control of their", Target: "behavior"
- Probe: "Achieving precise control of the behavior of unsupervised language models is difficult due to their", Target: "unsupervised training"

### Example 5
Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Contextualized Probes:
- Probe: "To steer unsupervised language models, existing methods collect", Target: "human labels"
- Probe: "Existing methods for gaining such steerability collect human labels of the relative quality of", Target: "model generations"
- Probe: "Existing methods align unsupervised language models by fine-tuning on human", Target: "preferences"
"""
    prompt['user'] = f"""### Input\nParagraph: {paragraph}\n\nSentence: {sentence}\n\n"""
    if len(paragraph.strip()) < 50:
        return None
    return utils.query_llm(prompt, model='gpt-5')

# Process first row only for testing
first_row = paper_df_validated.iloc[2]
atomic_facts = extract_atomic_facts(first_row['raw_knowledge_statement'], first_row['paragraph'])

print(f"Extracted atomic facts for first row: {atomic_facts}")

Extracted atomic facts for first row: - Probe: To steer unsupervised language models, existing methods collect, Target: human labels
- Probe: Existing methods for steering unsupervised language models collect human labels of the relative quality of, Target: model generations
- Probe: Existing methods align unsupervised language models by fine-tuning them to match human, Target: preferences
- Probe: To align unsupervised language models with human preferences, many approaches use reinforcement learning from human feedback, abbreviated as, Target: RLHF
- Probe: For outputs produced by language models, researchers collect labels from humans that assess the, Target: relative quality


In [162]:
paper_df_suitable

,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement,latex_percentage,is_suitable,unsuitable_condition,title
0,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,2.222222,True,NaN,Direct Preference Optimization: Your Language ...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,2.074689,True,NaN,Direct Preference Optimization: Your Language ...
2,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ...",1.773050,True,NaN,Direct Preference Optimization: Your Language ...
3,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...,0.833333,True,NaN,Direct Preference Optimization: Your Language ...
4,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit...",7.058824,True,NaN,Direct Preference Optimization: Your Language ...
...,...,...,...,...,...,...,...,...,...,...
148,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Rather than coercing the preference learning p...,4.244032,True,NaN,Direct Preference Optimization: Your Language ...
149,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","With virtually no tuning of hyperparameters, D...",1.687764,True,NaN,Direct Preference Optimization: Your Language ...
150,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Our initial results suggest that DPO policies ...,2.290076,True,NaN,Direct Preference Optimization: Your Language ...
151,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra...",3.208556,True,NaN,Direct Preference Optimization: Your Language ...


You will be given two inputs, a section of an academic paper for context and a single sentence drawn from that section. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

### Step 1: Target Extraction
First, identify the key central information of a sentence i.e. the targets. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in synctatic positions such as the main predicate, the subjective complement, or the object of the preposition. These are merely heuristics and the centrality of the phrase is much more important. Avoid the subject (e.g. methodology name, acronyms, etc.) of the sentence as a target if the subject is repeated often in the context. Apply this thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only *1-2* words and should not be a mathematical expression in LaTeX. Err on the side of extracting more targets than less.

### Step 2: Rewriting Facts that end with the Targets
Second, for each target, we will rewrite the sentence to become a 1) fully self-contained and atomic fact that 2) *ends* with the target. Use the surrounding paragraph to supply whatever context is needed to make the fact standalone. The fact should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that contextualized, self-contained, atomic facts ends with the targets. The most important rule is that the fact *must end with the target*. If this is impossible for some targets, then discard those facts at this step.

### Step 3: Reflect, Filter, and Refine
For each of the facts, consider if it's clear from the sentence. If it's not actually obvious from the sentence, we should not have extracted it. If there are many targets, consider if some of the targets are insignificant. The fact must end with the target, or it should be discarded otherwise. Considering these aspects, filter out appropriate targets and facts.

For the remaining facts, consider these conditions:
- **Self-Contained:** Is the sentence **fully understandable on its own**? 
- **Accuracy:** Does the rewritten fact **preserve the meaning** of the original? Is the fact clear from the sentence?
- **Fact Quality:** Does the fact flow naturally to end with the target? Is it written clearly?

Refine the facts to optimize for the above conditions. For instance, try to ensure that the content/information in the fact is similar to the phrasing of the original sentence.

### Step 4: Last Checks
- If there are mathematical expressions that are being referenced, check the surrounding context and include the definitions, equations, theorems, and givens that are being referenced to fully contextualize the math. Again, citing definitions are not enough, but they should be restated.
- For experimental details, ensure that the details are fully contextualized since experiments, in particular, only make sense in the context and scope of the paper. 

### Final Instructions

I've provided some demonstrations below which are some concrete examples that can guide your final output format. Before outputting the final format, think carefully through this task, following the step-by-step instructions outlined above, and provide your reasoning before the final extraction.

# Demonstrations
The next few demonstrations will be based on the following text. 

Context: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

### Example 1
Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "difficult
- "behavior"
- "unsupervised training"

Rewritten Facts:
- "world knowledge": "Large-scale unsupervised LMs learn broad world knowledge" 
- "reasoning skills": "Large-scale unsupervised LMs learn reasoning skills" 
- "precise control": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, their behavior is difficult to precisely control"
- "difficult": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, achieving precise control of their behavior is difficult"
- "behavior": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, it is difficult to achieve precise control of their behavior"
- "unsupervised training": "Achieving precise control of the behavior of unsupervised language models is difficult due to their unsupervised training"

### Example 2
Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Rewritten Facts:
- "human labels": "To steer unsupervised language models, existing methods collect human labels"
- "model generations": "Existing methods for steering unsupervised language models collect human labels of the relative quality of model generations"
- "preferences": "Existing methods align unsupervised language models by fine-tuning on human preferences"

###
Context: "\\title{{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}}\n\\subsection{{Can DPO scale to real preference datasets?}}\nNext, we evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on human preferences to provide more effective summaries. We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set. The completions for all methods are sampled at temperatures varying from 0.0 to 1.0, and the win rates are shown in Figure~\ref{fig:frontier-tldr-main} (right). DPO, PPO and Preferred-FT all fine-tune the same GPT-J SFT model\footnote{\url{https://huggingface.co/CarperAI/openai_summarize_tldr_sft}}. We find that DPO has a win rate of approximately 61\% at a temperature of 0.0, exceeding the performance of PPO at ~57\% at its optimal sampling temperature of 0.0. DPO also achieves a higher maximum win rate compared to the best of $N$ baseline. We note that we did not meaningfully tune DPO's $\beta$ hyperparameter, so these results may underestimate DPO's potential. Moreover, we find DPO to be much more robust to the sampling temperature than PPO, the performance of which can degrade to that of the base GPT-J model at high temperatures. Preferred-FT does not improve significantly over the SFT model. We also compare DPO and PPO head-to-head in human evaluations in Section~\ref{sec:human-judgments}, where DPO samples at temperature 0.25 were preferred 58\% times over PPO samples at temperature 0."

Sentence: "We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Targets:
- "TL;DR summarization"
- "reference completions"

Facts:
- "To evaluate the fine-tuning performance of DPO on summarization against other methods, completions are sampled on the test split of the dataset named TL;DR summarization"
- "The fine-tuning performance of DPO and other methods on summarization are evaluated by sampling completions on the test split the TL;DR summarization dataset and computing the average win rate against reference completions"


Context: "\\title{{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}}\n\\subsection{{Can DPO scale to real preference datasets?}}\nNext, we evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on human preferences to provide more effective summaries. We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set. The completions for all methods are sampled at temperatures varying from 0.0 to 1.0, and the win rates are shown in Figure~\ref{fig:frontier-tldr-main} (right). DPO, PPO and Preferred-FT all fine-tune the same GPT-J SFT model\footnote{\url{https://huggingface.co/CarperAI/openai_summarize_tldr_sft}}. We find that DPO has a win rate of approximately 61\% at a temperature of 0.0, exceeding the performance of PPO at ~57\% at its optimal sampling temperature of 0.0. DPO also achieves a higher maximum win rate compared to the best of $N$ baseline. We note that we did not meaningfully tune DPO's $\beta$ hyperparameter, so these results may underestimate DPO's potential. Moreover, we find DPO to be much more robust to the sampling temperature than PPO, the performance of which can degrade to that of the base GPT-J model at high temperatures. Preferred-FT does not improve significantly over the SFT model. We also compare DPO and PPO head-to-head in human evaluations in Section~\ref{sec:human-judgments}, where DPO samples at temperature 0.25 were preferred 58\% times over PPO samples at temperature 0."

Sentence: "We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Targets:
- "TL;DR summarization"
- "reference completions"

Facts:
- "To evaluate the fine-tuning performance of DPO on summarization against other methods, completions are sampled on the test split of the dataset named TL;DR summarization"
- "The fine-tuning performance of DPO and other methods on summarization are evaluated by sampling completions on the test split the TL;DR summarization dataset and computing the average win rate against reference completions"


In [71]:
import concurrent.futures

# Context: "Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning in language models.
# \begin{enumerate}[topsep=1pt,itemsep=0ex]%
#     \item First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps.
#     \item Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question).
#     \item Third, chain-of-thought reasoning can be used for tasks such as math word problems, commonsense reasoning, and symbolic manipulation, and is potentially applicable (at least in principle) to any task that humans can solve via language.
#     \item Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting.
# \end{enumerate}"

# ### Example 1
# Sentence: "First, chain of thought, in principle, allows models to decompose multi-step problems into intermediate steps, which means that additional computation can be allocated to problems that require more reasoning steps." 

# Targets:
# - "intermediate steps"
# - "reasoning steps"

# Rewritten Facts:
# - "intermediate steps": "Chain of thought allows models to decompose multi-step problems into intermediate steps"
# - "reasoning steps": "By decomposing multi-step problems into intermediate steps, chain of thought allows additional computation to be allocated to problems that require more reasoning steps"

# ### Example 2
# Sentence: "Second, a chain of thought provides an interpretable window into the behavior of the model, suggesting how it might have arrived at a particular answer and providing opportunities to debug where the reasoning path went wrong (although fully characterizing a model's computations that support an answer remains an open question)."

# Targets:
# - "interpretable window"
# - "reasoning path"

# Rewritten Facts:
# - "interpretable window": "Chain-of-thought shows how the model formed an answer and provides opportunities to debug mistakes in the reasoning path"
# - "reasoning path": "The behavior of language models can be difficult to characterize but chain-of-thought provides an interpretable window"

# # More Demonstrations
# The next few examples will be based on the following text for context. 

# ### Example 3
# Sentence: "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting."

# Targets:
# - "including examples"
# - "elicited"

# Contextualized Targets:
# - "Chain-of-thought reasoning can be elicited in sufficiently large off-the-shelf language models simply by including examples"
# - "By including examples of chain of thought sequences, chain-of-thought reasoning in sufficiently large off-the-shelf language models can be readily elicited"


# ### Example 1
# Context: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training."

# Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

# Targets:
# - "world knowledge"
# - "reasoning skills"
# - "precise control"
# - "difficult
# - "behavior"
# - "unsupervised training"

# Rewritten Facts:
# - "world knowledge": "Large-scale unsupervised LMs learn broad world knowledge" 
# - "reasoning skills": "Large-scale unsupervised LMs learn reasoning skills" 
# - "precise control": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, their behavior is difficult to precisely control"
# - "difficult": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, achieving precise control of their behavior is difficult"
# - "behavior": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, it is difficult to achieve precise control of their behavior"
# - "unsupervised training": "Achieving precise control of the behavior of unsupervised language models is difficult due to their unsupervised training"

### Step 4: Reflect and Refine
# For the remaining facts, consider these conditions:
# - *Self-Contained:* Is the sentence fully understandable on its own? 
# - *Accuracy:* Does the rewritten fact preserve the meaning of the original? Try to ensure that the content of the fact is similar to the phrasing in the original sentence.
# - *Fact Quality:* Is the fact written in a natural manner? Is the target unnnaturally forced to be at the end e.g. repeated just so the sentence ends with it?
# - *Target Validity:* Some targets may be impossible to place at the end of the fact. Consider if the *phrase that the fact currently ends with* can be considered as the target instead. If it cannot, then discard the target and output "NA".

# Please further refine the facts to optimize for the above conditions. We want our facts to be of the upmost quality regarding these conditions. 

# - *Target Validity:* Some targets may be impossible to place at the end of the fact. Consider if the *phrase that the fact currently ends with* can be considered as the target instead. If it cannot, then discard the target and output "NA".

def extract_atomic_facts(first_row):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You will be given two inputs, a section of an academic paper for context and a single sentence drawn from that section. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

### Step 1: Target Extraction
First, identify the key central information of a sentence i.e. the targets. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in synctatic positions such as the main predicate, the subjective complement, or the object of the preposition. These are merely heuristics and the centrality of the phrase is much more important. Avoid the subject (e.g. methodology name, acronyms, etc.) of the sentence as a target if it is repeated often in the context. Apply this thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only *1-3* words and should not be a mathematical expression in LaTeX.

### Step 2: Write Facts that End with the Targets
For each target, you will take the parts of the sentence relevant to the target, and write a fact contained in the provided sentence. The fact should be obvious from the sentence. This fact must *end* with the target. This shouldn't be forced and the paraphrasing should naturally end with its target. Please do not just repeat the target at the end because there's no other way to end with it. If this is impossible for some targets, then discard those targets at this step.

### Step 3: Contextualize The Facts
Then, for each target, you will rewrite the facts to become a fully self-contained and atomic fact that still *ends* with its corresponding target. The fact doesn't need to be a single sentence; if it's more natural to break it up, use one sentence to set the context, and one more sentence to state the fact. In either case, use the provided *context* to supply whatever information is needed to make the fact standalone. The fact should be understandable on its own, without the context, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the context was about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. At the same time, the fact should preserve the original meaning in the sentence so use language, LaTeX syntax, and phrasing that is similar to the original sentence. Again, ensure that each contextualized, self-contained, and atomic fact ends with its target.

### Step 4: Reflect and Repeat
For each fact, generate 4 additional versions of the fact, remembering these core conditions:
- *Self-Contained:* Is the sentence fully understandable on its own? 
- *Accuracy:* Does the rewritten fact preserve the meaning of the original? Ensure that the content of the fact is similar to the phrasing in the original sentence.
- *Fact Quality:* Is the fact written in a natural manner? Is the target unnnaturally forced to be at the end e.g. repeated just so the sentence ends with it?
- *Math Contextualization:* If there are mathematical expressions that are being referenced, check the surrounding context and include the definitions, equations, theorems, and givens that are being referenced to fully contextualize the math. Citing definitions are not enough, but they should be restated.
- *LaTeX Formatting:* The rewritten facts *MUST* follow the same syntax and formatting of LaTeX as the original sentence.
- *Experimental Contextualization:* For experimental details, ensure that the details are fully contextualized since experiments, in particular, only make sense in the context and scope of the paper. 

### Final Instructions

I've provided some demonstrations below which are some concrete examples that can guide your reasoning and final output format. 

Before outputting your final response, think carefully through this task, following the step-by-step instructions outlined above and provide your reasoning. After each step, show the current status of the facts and targets.

# Demonstrations

### Example 1
Context: "\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Targets:
- "human labels"
- "model generations"
- "preferences"

Facts:
- "human labels": "To steer unsupervised language models, existing methods collect human labels"
- "model generations": "Existing methods for steering unsupervised language models collect human labels of the relative quality of model generations"
- "preferences": "Existing methods align unsupervised language models by fine-tuning on human preferences"

### Example 2
Context: "\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\n\\subsection{Can DPO scale to real preference datasets?}\nNext, we evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on human preferences to provide more effective summaries. We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Sentence: "We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Targets:
- "TL;DR summarization"
- "reference completions"

Facts:
- "TL;DR summarization": "To evaluate the fine-tuning performance of DPO on summarization against other methods, completions are sampled on the test split of the dataset named TL;DR summarization"
- "reference completions": "The fine-tuning performance of DPO and other methods on summarization are evaluated by sampling completions on the test split the TL;DR summarization dataset and computing the average win rate against reference completions"

### Example 3
Context: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training."

Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

Targets:
- "world knowledge"
- "reasoning skills"
- "precise control"
- "difficult
- "behavior"
- "unsupervised training"

Rewritten Facts:
- "world knowledge": "Large-scale unsupervised LMs learn broad world knowledge" 
- "reasoning skills": "Large-scale unsupervised LMs learn reasoning skills" 
- "precise control": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, their behavior is difficult to precisely control"
- "difficult": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, achieving precise control of their behavior is difficult"
- "behavior": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, it is difficult to achieve precise control of their behavior"
- "unsupervised training": "Achieving precise control of the behavior of unsupervised language models is difficult due to their unsupervised training"
"""
    context = first_row['subsection_text'].strip().split(first_row['raw_knowledge_statement'])[0].strip()
    prompt['user'] = f"""### Context\n\\title{{{first_row['title']}}}\n\n\subsection{{{first_row['subsection'].strip()}}}\n{context}\n\n### Sentence\n{first_row['raw_knowledge_statement'].strip()}"""
    #print(prompt['user'])
    if len(first_row['raw_knowledge_statement'].strip()) < 50:
        return None, None
    output1 = utils.query_llm(prompt, model='o4-mini')
    output2 = utils.query_llm(prompt, model='o4-mini')
    return output1, output2
# # Process all rows
from tqdm import tqdm

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    raw_extracted_facts = list(tqdm(executor.map(extract_atomic_facts, [paper_df_suitable.iloc[i] for i in range(len(paper_df_suitable))]), total=len(paper_df_suitable)))

# Separate the two outputs into different columns
output1_list = [result[0] if result is not None else None for result in raw_extracted_facts]
output2_list = [result[1] if result is not None else None for result in raw_extracted_facts]

# Store in dataframe
paper_df_suitable['raw_extracted_facts_1'] = output1_list
paper_df_suitable['raw_extracted_facts_2'] = output2_list

print(f"Processed {len(raw_extracted_facts)} rows")
print(f"Sample result 1: {raw_extracted_facts[0][0] if raw_extracted_facts[0] is not None else None}")
print(f"Sample result 2: {raw_extracted_facts[0][1] if raw_extracted_facts[0] is not None else None}")

# Process just one row for testing
# first_row = paper_df_suitable.iloc[44]
# result = extract_atomic_facts(first_row)

# print(f"Processed 1 row")
# print(f"Title: {first_row['title']}")
# print(f"Subsection: {first_row['subsection_text'].strip().split(first_row['raw_knowledge_statement'])[0].strip()}")
# print(f"Sentence: {first_row['raw_knowledge_statement']}")

# print(f"Sample result: {result}")


# ### Example 3
# Context: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training."

# Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

# Targets:
# - "world knowledge"
# - "reasoning skills"
# - "precise control"
# - "difficult
# - "behavior"
# - "unsupervised training"

# Rewritten Facts:
# - "world knowledge": "Large-scale unsupervised LMs learn broad world knowledge" 
# - "reasoning skills": "Large-scale unsupervised LMs learn reasoning skills" 
# - "precise control": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, their behavior is difficult to precisely control"
# - "difficult": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, achieving precise control of their behavior is difficult"
# - "behavior": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, it is difficult to achieve precise control of their behavior"
# - "unsupervised training": "Achieving precise control of the behavior of unsupervised language models is difficult due to their unsupervised training"

### Step 4: Filter
# Is the fact clear? If it's not actually obvious from the sentence, we shouldn't have extracted the target. If there are many targets, consider if some of the targets are insignificant. Considering these aspects, filter out some targets and facts.


100%|██████████| 150/150 [15:00<00:00,  6.00s/it]

Processed 150 rows
Sample result 1: Here is the step-by-step application of the procedure:

Step 1: Target Extraction  
We look for the core new information in the sentence (avoiding repeated subjects like “existing methods” or “unsupervised LM”).  
• “human labels” – the data collected to gain steerability  
• “model generations” – the outputs whose quality is judged  
• “preferences” – the signals used to align the LM  
• “reinforcement learning from human feedback” – the common fine-tuning paradigm  

Step 2: Initial Atomic Facts (ending with each target)  
1. Existing methods for gaining steerability collect human labels  
2. Existing methods collect human labels of model generations  
3. Existing methods fine-tune the unsupervised LM to align with preferences  
4. Existing methods often fine-tune with reinforcement learning from human feedback  

Step 3: Contextualized, Self-Contained Facts (still ending with each target)  
1. To steer large-scale unsupervised language models, exi

In [72]:
# Print 25 random outputs from the extracted facts
import random

# Get valid indices (where raw_extracted_facts is not None)
valid_indices = [i for i, facts in enumerate(raw_extracted_facts) if facts is not None]

# Sample 25 random indices
sample_indices = sorted(random.sample(valid_indices, min(20, len(valid_indices))))

print(f"Showing {len(sample_indices)} random outputs from extracted facts:\n")
print("=" * 80)

for i, idx in enumerate(sample_indices, 1):
    row = paper_df_suitable.iloc[idx]
    facts = raw_extracted_facts[idx]
    
    print(f"\n{i}. Row {idx}")
    print(f"Title: {row['title']}")
    print(f"Subsection: {row['subsection']}")
    print(f"Original sentence: {row['raw_knowledge_statement']}")
    print(f"Extracted facts: {facts}")
    print("-" * 50)


Showing 20 random outputs from extracted facts:


1. Row 1
Title: Direct Preference Optimization: Your Language Model is Secretly a Reward Model
Subsection: No Subsection
Original sentence: However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.
Extracted facts: ('Step 1: Target Extraction  \nThe sentence describes four central pieces of information:  \n- that RLHF is a “complex procedure”  \n- that RLHF is an “unstable procedure”  \n- that RLHF first fits a “reward model”  \n- that RLHF then performs “fine-tuning”  \n\nStep 2: Initial Facts Ending with the Targets  \n1. Reinforcement learning from human feedback (RLHF) is a complex procedure.  \n2. Reinforcement learning from human feedback (RLHF) is often an unstable procedure.  \n3. Reinforcement learning from

### [OLD] 4. Filter and Refine Probes

In [325]:
import concurrent.futures
 
def validate_atomic_facts(previous_output):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You are a quality control assistant. Your task is to review and refine the output of a previous process. The original task was to deconstruct a sentence from a paper into self-contained, atomic facts that ends with their corresponding targets.

## Instructions
Approach this task carefully, following the step-by-step guidelines provided below and provide your reasoning before the final output.

First, extract the finalized list of `(target, rewritten_sentence)` pairs from the output of the previous process.

Then, for each pair of target and fact, carefully perform the following checks.
- *Target Validity:*  Does the fact properly end with its corresponding target? Does the target only appear once in the fact?
- *Fact Quality:* Is the writing of the fact natural? Is it fully understandable on its own?

If any pairs fail the checks, discard them.

Lastly, output the filtered list of `(target, rewritten_sentence)` pairs. Copy the facts exactly as they are, including any LaTeX formatting. Make sure that any mathematical expressions or notations are written in the same LaTeX format as provided."""
    prompt['user'] = f"""# Output of Previous Process\n\n{previous_output}"""
    
    response = utils.query_llm(prompt, model='gpt-4.1-mini', system_prompt_included=True)
    return response
    
def format_atomic_facts(previous_output):
    prompt = {}
    prompt['system'] = """Format the final list of targets and their corresponding facts into JSON with the following key. Copy the facts exactly as they are, including any LaTeX formatting.
- "pairs": list of dicts i.e ("target": target, "rewritten_fact": rewritten fact)"""
    prompt['user'] = f"""### List of Facts\n\n{previous_output}"""
    
    response = utils.query_llm(prompt, model='gpt-4.1-nano', return_json=True, system_prompt_included=True)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON: {response}")
        return None
    
print(paper_df_suitable.iloc[80]['raw_extracted_facts'])
filtering = validate_atomic_facts(paper_df_suitable.iloc[80]['raw_extracted_facts'])
print(filtering)
formatted_facts = format_atomic_facts(filtering)
print(formatted_facts)
    

Let's proceed step by step as instructed.

---

## Step 1: Target Extraction

Sentence:  
"We can alternatively view Theorem~\ref{thm:main} as specifying exactly which reward function within each equivalence class the DPO reparameterization selects, that is, the reward function satisfying:
\begin{equation}\label{eq:lag_p}
     \sum_{y}\underbrace{\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)}_{=\pi(y\mid x)\text{, using Thm.~\ref{thm:main} reparam.}} = 1,
\end{equation}
i.e., $\pi(y\mid x)$ is a valid distribution (probabilities are positive and sum to 1)."

Key, central information:
- "reward function"
- "equivalence class"
- "DPO reparameterization"
- "valid distribution"

Let's check which of these are central. The sentence is about Theorem~\ref{thm:main} and how it specifies which reward function (within an equivalence class) is selected by the DPO reparameterization, namely the one that makes $\pi(y|x)$ a valid distribution. The main predicates are "specifying", "selects"

### 4. Filter and Refine Probes

In [ ]:
import concurrent.futures
from importlib import reload 
import utils.utils as utils
reload(utils)

def validate_atomic_facts(row):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You are a quality control assistant. Your task is to review and refine the outputs of a previous process. The original task was to deconstruct a sentence from a paper into self-contained, atomic facts that ends with their corresponding targets.

## Instructions
Approach this task carefully, following the step-by-step guidelines provided below.

First, extract the finalized list of `(target, rewritten_sentence)` pairs from the outputs of the previous process. There should be 10 in total, including the original and the additional paraphrased versions of the fact.

For each of the targets, choose the 3 best rewritten facts, with these aspects in mind:
- *Target Validity:*  Does the fact naturally end with its corresponding target? Does the target only appear once in the fact?
- *Fact Quality:* Is the fact clear and written well? 
- *Self-Contained:* Is the fact understandable on its own without any context?
- *Accuracy:* Does the rewritten fact properly capture the information and phrasing of the original? 

Furthermore, once you have chosen the 3 best versions of the fact, PLEASE rewrite the mathematical notation and expressions in proper LaTeX format:
- Use the same LaTeX syntax, spacing, and formatting as the provided context. 
- Do not precede parentheses with a backslash e.g. \\( and make sure to surround algebra, expressions, and LaTeX commands e.g. \\pi with $.
- Translate the math without any styling such as \displaystyle
- Leave numbers alone; don't surround them with $.

Lastly, output the filtered list of `(target, rewritten_sentence)` pairs. Copy the facts exactly as they are."""
    sentence = row['raw_knowledge_statement']
    previous_output_1 = row['raw_extracted_facts_1']
    previous_output_2 = row['raw_extracted_facts_2']
    context = row['subsection_text'].strip().split(row['raw_knowledge_statement'])[0].strip()

    prompt['user'] = f"""### Original Context of Sentence\n{context + sentence}\n\n### Output 1 of Previous Process\n{previous_output_1}\n\n### Output 2 of Previous Process\n{previous_output_2}"""
    
    response = utils.query_gpt(prompt, model='o4-mini', system_prompt_included=True, reasoning_effort='high')
    return response
    
def format_atomic_facts(previous_output):
    prompt = {}
    prompt['system'] = r"""Format the final list of targets and their corresponding facts into JSON with the following key. Copy the facts exactly as they are, including the LaTeX formatting exactly as it was provided. DO NOT precede parentheses with a backslash \ and make sure to surround algebra or LaTeX commands e.g. \pi with $. Leave numbers alone; don't surround them with $. Only use a single backslash for LaTeX commands e.g. $r_{\phi}(x, y)$.
- "pairs": list of dicts i.e ("target": target, "rewritten_fact": rewritten fact)"""
    prompt['user'] = f"""### List of Facts\n\n{previous_output}"""
    
    response = utils.query_gpt(prompt, model='o4-mini', return_json=True, system_prompt_included=True, reasoning_effort='low')
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON: {response}")
        return None
    
print(paper_df_suitable.iloc[85]['paragraph'])
print(paper_df_suitable.iloc[85]['raw_extracted_facts_1'])
filtering = validate_atomic_facts(paper_df_suitable.iloc[85])
print(filtering)
formatted_facts = format_atomic_facts(filtering)
print(formatted_facts)
    

\begin{lemma}\label{lemma:same_policy}
    Two reward functions from the same equivalence class induce the same optimal policy under the constrained RL problem.
\end{lemma}
The proofs are straightforward and we defer them to Appendix \ref{app:lemma1}. The first lemma is a well-known under-specification issue with the Plackett-Luce family of models \cite{plackett1975analysis}. Due to this under-specification, we usually have to impose additional identifiability constraints to achieve any guarantees on the MLE estimates from Eq. \ref{eq:reward_model} \cite{bong2022generalized}. The second lemma states that all reward functions from the same class yield the same optimal policy, hence for our final objective, we are only interested in recovering an arbitrary reward function from the optimal class. We prove the following Theorem in Appendix~\ref{app:thm1}:
\begin{theorem}\label{thm:main}
    Under mild assumptions, all reward classes consistent with the Plackett-Luce (and Bradley-Terry in p

In [146]:
def process_row(row):
    """Process a single row for validation."""
    return format_atomic_facts(validate_atomic_facts(row))

# Process all rows in parallel
with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
    validated_results = list(tqdm(
        executor.map(process_row, [row for _, row in paper_df_suitable.iterrows()]),
        total=len(paper_df_suitable)
    ))

# Add results as a new column
paper_df_suitable['validated_atomic_pairs'] = validated_results

print(f"Processed {len([r for r in validated_results if r is not None])} rows with validated pairs")

100%|██████████| 150/150 [06:50<00:00,  2.73s/it]

Processed 150 rows with validated pairs


In [147]:
# Display 10 sample atomic pairs for review
print("Sample Validated Atomic Pairs:")
print("=" * 50)

sample_count = 0
for idx, row in paper_df_suitable.iterrows():
    if row['validated_atomic_pairs'] is not None and sample_count < 100:
        pairs = row['validated_atomic_pairs'].get('pairs', [])
        if pairs:
            print(f"\nRow {idx}:")
            print(f"Original Statement: {row['raw_knowledge_statement']}")
            print(f"Section: {row['section']} - {row['subsection']}")
            print(f"Number of atomic pairs: {len(pairs)}")
            print("Atomic Pairs:")
            for i, pair in enumerate(pairs):
                print(f"  {i+1}. Target: '{pair['target']}'")
                print(f"     Probe: {pair['rewritten_fact']}")
            print("-" * 40)
            sample_count += 1
    
    if sample_count >= 100:
        break

if sample_count == 0:
    print("No validated atomic pairs found in the data.")


Sample Validated Atomic Pairs:

Row 0:
Original Statement: Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
Section: Title/Abstract - No Subsection
Number of atomic pairs: 6
Atomic Pairs:
  1. Target: 'human labels'
     Probe: To achieve steerability of unsupervised language models, existing methods collect human labels
  2. Target: 'human labels'
     Probe: Existing methods aiming to steer unsupervised language models gather human labels
  3. Target: 'human labels'
     Probe: In order to control the behavior of unsupervised language models, current approaches collect human labels
  4. Target: 'model generations'
     Probe: For steering unsupervised language models, existing methods collect human labels about the relative quality of model generations
  5. Target: 'model generations'
     Prob

### 5. Check Target is truly at end of the sentence -> Split Fact into Fact and Target

In [90]:
# Explode the validated_atomic_pairs into separate rows
rows_for_df = []

for idx, row in paper_df_suitable.iterrows():
    if row['validated_atomic_pairs'] is not None:
        pairs = row['validated_atomic_pairs'].get('pairs', [])
        for pair in pairs:
            new_row = row.to_dict()
            new_row['target'] = pair['target']
            new_row['fact'] = pair['rewritten_fact']
            rows_for_df.append(new_row)

# Create new dataframe with exploded facts
paper_df_with_probes = pd.DataFrame(rows_for_df)

print(f"Exploded {len(paper_df_suitable)} rows with validated pairs into {len(paper_df_with_probes)} individual fact rows")


Exploded 150 rows with validated pairs into 976 individual fact rows


In [91]:
paper_df_with_probes = paper_df_with_probes[['section','subsection', 'section_text','subsection_text', 'raw_knowledge_statement', 'target', 'fact']]

In [95]:
# Fix facts that start with '(' by extracting the rightmost part after target + ','
facts_starting_with_paren = paper_df_with_probes[paper_df_with_probes['fact'].str.startswith('(')]
print(f"Found {len(facts_starting_with_paren)} facts starting with '(' - fixing these...")

for idx, row in facts_starting_with_paren.iterrows():
    target_with_comma = row['target'] + ','
    if target_with_comma in row['fact']:
        rightmost_part = row['fact'].split(target_with_comma)[-1].strip().strip('()')
        # Update the fact in the dataframe
        paper_df_with_probes.loc[idx, 'fact'] = rightmost_part
        print(f"Fixed fact at index {idx}: '{row['fact']}' -> '{rightmost_part}'")

print(f"Fixed {len(facts_starting_with_paren)} facts that started with '('")


Found 12 facts starting with '(' - fixing these...
Fixed fact at index 189: '(effective, In this paper, we compare Direct Preference Optimization (DPO) with existing fine‐tuning methods—specifically PPO‐based reinforcement learning from human feedback—on preference‐learning tasks such as sentiment modulation, summarization, and dialogue using language models with up to 6 billion parameters, and these experiments show that DPO is at least as effective)' -> 'In this paper, we compare Direct Preference Optimization (DPO) with existing fine‐tuning methods—specifically PPO‐based reinforcement learning from human feedback—on preference‐learning tasks such as sentiment modulation, summarization, and dialogue using language models with up to 6 billion parameters, and these experiments show that DPO is at least as effective'
Fixed fact at index 190: '(effective, Experimental evaluations of Direct Preference Optimization against PPO‐based reinforcement learning from human feedback on tasks inclu

In [96]:
# Check that facts end with targets (after stripping whitespace and punctuation)
import string

valid_facts = []
filtered_count = 0
dropped_examples = []

for idx, row in paper_df_with_probes.iterrows():
    fact = str(row['fact']).strip()
    target = str(row['target']).strip()
    
    # Remove punctuation from the end of fact for comparison
    fact_cleaned = fact.rstrip(string.whitespace).rstrip('.')
    
    # Check if fact ends with target
    if fact_cleaned.lower().endswith(target.lower()):
        valid_facts.append(True)
    else:
        valid_facts.append(False)
        filtered_count += 1
        dropped_examples.append({'fact': fact, 'target': target})

paper_df_with_probes['valid_fact'] = valid_facts

print(f"Filtered out {filtered_count} facts that don't end with their target")
print(f"Remaining valid facts: {len(paper_df_with_probes) - filtered_count}")

# Print some examples that were dropped
if dropped_examples:
    print("\nExamples of dropped fact-target pairs:")
    for i, example in enumerate(dropped_examples[:5]):  # Show first 5 dropped examples
        print(f"  {i+1}. Fact: '{example['fact']}'")
        print(f"     Target: '{example['target']}'")
        print()
        
# Filter to keep only valid facts
paper_df_with_probes = paper_df_with_probes[paper_df_with_probes['valid_fact']].copy()


Filtered out 53 facts that don't end with their target
Remaining valid facts: 923

Examples of dropped fact-target pairs:
  1. Fact: 'Through controlled experiments on model steerability, we find that Direct Preference Optimization is capable of fine-tuning LMs.'
     Target: 'fine-tune LMs'

  2. Fact: 'Large unsupervised language models are trained on massive human‐generated text and thus acquire a broad spectrum of capabilities. To ensure the resulting AI systems are safe, performant, and controllable, developers must select the model’s esired responses and behavior.'
     Target: 'desired responses and behavior'

  3. Fact: 'Because these models are trained on human text with mixed goals and skill levels, they can reproduce undesirable outputs. To avoid that and achieve safe, performant, and controllable AI, one must deliberately choose the model’s esired responses and behavior.'
     Target: 'desired responses and behavior'

  4. Fact: 'Large‐scale unsupervised LMs reflect a w

In [97]:
# Create probe column (fact minus target) using stripped, no punctuation versions
import string

probes = []
cleaned_facts = []
cleaned_targets = []

for idx, row in paper_df_with_probes.iterrows():
    fact = str(row['fact']).strip()
    target = str(row['target']).strip()
    
    # Clean fact and target by removing punctuation from the end
    fact_cleaned = fact.rstrip(string.punctuation + string.whitespace)
    target_cleaned = ' ' + target.rstrip(string.punctuation + string.whitespace)
    
    
    last_index = fact_cleaned.rfind(target_cleaned)
    if last_index != -1:
        probe = fact_cleaned[:last_index].strip()
        probes.append(probe)
        cleaned_facts.append(fact_cleaned)
        cleaned_targets.append(target_cleaned)
    else:
        print(fact)
        print(target)
        raise ValueError(f"Target {target_cleaned} not found in fact {fact_cleaned}")

paper_df_with_probes['probe'] = probes
paper_df_with_probes['fact'] = cleaned_facts
paper_df_with_probes['target'] = cleaned_targets

print(f"Created probe column by removing target from fact")
print(f"Final dataset shape: {paper_df_with_probes.shape}")
#paper_df_with_probes.drop(columns=['valid_fact'], inplace=True)
paper_df_with_probes.head(5)

Created probe column by removing target from fact
Final dataset shape: (923, 9)


,section,subsection,section_text,subsection_text,raw_knowledge_statement,target,fact,valid_fact,probe
0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,human labels,To achieve steerability of unsupervised langua...,True,To achieve steerability of unsupervised langua...
1,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,human labels,Existing methods aiming to steer unsupervised ...,True,Existing methods aiming to steer unsupervised ...
2,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,human labels,To guide unsupervised language models toward d...,True,To guide unsupervised language models toward d...
3,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,model generations,"For steering unsupervised language models, exi...",True,"For steering unsupervised language models, exi..."
4,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,model generations,To evaluate and steer unsupervised language mo...,True,To evaluate and steer unsupervised language mo...


### 6. Ensure tokenizing the target separately from the probe is fine

In [98]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")

contexts = paper_df_with_probes['probe'].tolist()
targets = paper_df_with_probes['target'].tolist()
facts = paper_df_with_probes['fact'].tolist()

print(f"--- Tokenizer and String Consistency Check ---")

string_mismatches = 0
token_mismatches = 0

for i, (context, target, fact) in enumerate(zip(contexts, targets, facts)):
    # --- String Level Check ---
    reconstructed_string = context + target
    string_match = reconstructed_string == fact
    
    if not string_match:
        string_mismatches += 1
        print(f"--- ❌ STRING MISMATCH: Sample #{i} ---")
        print(f"Context: '{context}'")
        print(f"Target:  '{target}'")
        print(f"Fact:    '{fact}'")
        print(f"Reconstructed: '{reconstructed_string}'")
        print(f"Match: {string_match}")
        print("-" * 30)
        continue
    
    # --- Tokenization Level Check ---
    # Method 1: Tokenize the fact directly
    tokenized_fact = tokenizer(fact, add_special_tokens=False, padding=False)['input_ids']
    
    # Method 2: Tokenize parts separately and concatenate
    tokenized_context = tokenizer(context, add_special_tokens=False, padding=False)['input_ids']
    tokenized_target = tokenizer(target, add_special_tokens=False, padding=False)['input_ids']
    tokenized_parts_combined = tokenized_context + tokenized_target

    # --- Comparison ---
    if tokenized_fact != tokenized_parts_combined:
        token_mismatches += 1
        print(f"--- ❌ TOKEN MISMATCH: Sample #{i} ---")
        print(f"Context: '{context}'")
        print(f"Target:  '{target}'")
        print(f"Fact:    '{fact}'")
        print(f"\nTokenizing fact directly:           {tokenized_fact} (Length: {len(tokenized_fact)})")
        print(f"Tokenizing parts and concatenating: {tokenized_parts_combined} (Length: {len(tokenized_parts_combined)})")
        
        # Find differing positions
        min_len = min(len(tokenized_fact), len(tokenized_parts_combined))
        diff_positions = []
        for pos in range(min_len):
            if tokenized_fact[pos] != tokenized_parts_combined[pos]:
                diff_positions.append(pos)
        
        if diff_positions:
            print(f"\nFirst differing positions: {diff_positions[:5]}")
            for pos in diff_positions[:3]:
                fact_token = tokenizer.decode([tokenized_fact[pos]])
                parts_token = tokenizer.decode([tokenized_parts_combined[pos]])
                print(f"  Position {pos}: fact='{fact_token}' vs parts='{parts_token}'")
        
        print("-" * 30)

print(f"\nSummary:")
print(f"String mismatches: {string_mismatches}")
print(f"Token mismatches: {token_mismatches}")
print(f"Total samples: {len(contexts)}")


--- Tokenizer and String Consistency Check ---

Summary:
String mismatches: 0
Token mismatches: 0
Total samples: 923


### 7. Save the Probes

In [100]:
paper_df_with_probes.reset_index(drop=True, inplace=True)
paper_df_with_probes.drop(columns=['valid_fact'], inplace=True)
paper_df_with_probes.to_csv('../../data/arxiv/DPO_knowledge_probes_v5.csv', index=False)

In [101]:
print("All facts in paper_df_with_probes:")
for i, fact in enumerate(paper_df_with_probes['fact']):
    print(f"{i}: {fact}")


All facts in paper_df_with_probes:
0: To achieve steerability of unsupervised language models, existing methods collect human labels
1: Existing methods aiming to steer unsupervised language models gather human labels
2: To guide unsupervised language models toward desired behaviors, researchers collect human labels
3: For steering unsupervised language models, existing methods collect human labels about the relative quality of model generations
4: To evaluate and steer unsupervised language models, methods collect human labels on the relative quality of model generations
5: Steerability in unsupervised language models is achieved by collecting human labels regarding the relative quality of model generations
6: To gain steerability over unsupervised language models, existing methods fine-tune the models to align with human preferences
7: Steering unsupervised language models involves fine-tuning them so that they align with human preferences
8: To control unsupervised language models, 

### Appendix A: Examining Tokenization of "X." vs X"

In [9]:
# Check tokenization of "Z(x)" variants
import transformers

# Load the tokenizer (using the same one as in your model)
tokenizer = transformers.AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")

# Test strings
test_strings = ["Z(x)", "Z(x).", "Z(x) )", "\\beta \\log Z(x)"]

print("Checking tokenization of Z(x) variants:")
for test_str in test_strings:
    tokens = tokenizer.tokenize(test_str)
    token_ids = tokenizer.encode(test_str, add_special_tokens=False)
    print(f"'{test_str}' -> tokens: {tokens} -> ids: {token_ids}")

# Check if "Z(x)" is a common prefix
zx_tokens = tokenizer.tokenize("Z(x)")
zx_ids = tokenizer.encode("Z(x)", add_special_tokens=False)

print(f"\nBase 'Z(x)' tokens: {zx_tokens} -> ids: {zx_ids}")

for test_str in ["Z(x).", "Z(x) )"]:
    test_tokens = tokenizer.tokenize(test_str)
    test_ids = tokenizer.encode(test_str, add_special_tokens=False)
    
    # Check if Z(x) tokens are a prefix
    is_prefix = len(zx_ids) <= len(test_ids) and test_ids[:len(zx_ids)] == zx_ids
    print(f"'{test_str}' has 'Z(x)' as prefix: {is_prefix}")


Checking tokenization of Z(x) variants:
'Z(x)' -> tokens: ['Z', '(x', ')'] -> ids: [57, 2120, 8]
'Z(x).' -> tokens: ['Z', '(x', ').'] -> ids: [57, 2120, 570]
'Z(x) )' -> tokens: ['Z', '(x', ')', 'Ġ)'] -> ids: [57, 2120, 8, 883]
'\beta \log Z(x)' -> tokens: ['\\', 'beta', 'Ġ\\', 'log', 'ĠZ', '(x', ')'] -> ids: [59, 19674, 1144, 848, 1901, 2120, 8]

Base 'Z(x)' tokens: ['Z', '(x', ')'] -> ids: [57, 2120, 8]
'Z(x).' has 'Z(x)' as prefix: False
'Z(x) )' has 'Z(x)' as prefix: True


In [31]:
# Check tokenization of the mathematical expression
test_expression = r"\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)}$."

print(f"Testing tokenization of: {test_expression}")
tokens = tokenizer.tokenize(test_expression)
token_ids = tokenizer.encode(test_expression, add_special_tokens=False)
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(tokens)}")


Testing tokenization of: \beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)}$.
Tokens: ['\\', 'beta', 'Ġ\\', 'log', 'Ġ\\', 'frac', '{\\', 'pi', '_r', '(y', '\\', 'mid', 'Ġx', ')}', '{\\', 'pire', 'f', '(y', '\\', 'mid', 'Ġx', ')}', '$.']
Token IDs: [59, 19674, 1144, 848, 1144, 38118, 36802, 2554, 1745, 7166, 59, 16497, 865, 9317, 36802, 23772, 69, 7166, 59, 16497, 865, 9317, 13244]
Number of tokens: 23


# Appendix B: Old Prompts


You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

### Step 1: Target Extraction
First, identify the key central information of a sentence i.e. the targets. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For more complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words and should not be a mathematical expression in LaTeX. Err on the side of extracting more targets than less.

### Step 2: Rewriting Facts with Target
Second, for each target, we will rewrite the sentence to become a 1) fully self-contained and atomic fact that 2) ends with the target. The fact should only contain information *relevant* to the target.
Use the surrounding paragraph to supply whatever context is needed to make the fact standalone. The fact should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that contextualized, self-contained, atomic facts ends with the targets. If this is impossible for a target, then discard the target and output "NA" for the target.

### Step 3: Reflect and Refine

There are a few cases to look out for in particular. If there are mathematical expressions that are being referenced, check the surrounding context and include the original definitions and givens to fully contextualize the sentence so that it can stand alone. If the references cannot be found, discard all probes for this sentence and output "NA". For experimental details, ensure that the details are fully contextualized since experiments, in particular, only make sense in the context and scope of the paper. 

Furthermore, for each fact, check the following:
* **Target Placement:** Does the sentence actually end with the target word(s)? 
* **Self-Contained:** Is the sentence **fully understandable on its own**? 
* **Accuracy:** Does the rewritten fact **preserve the exact meaning** of the original without distortion? Note some facts preserve only parts of the original sentence; this is fine. It should be still be accurate to the original meaning.
* **Fact Quality:** Is the fact clearly obvious from the sentence? Is the construction and writing of the fact natural and unforced? Does the fact flow naturally to end with the target?

Refine the facts to satisfy these conditions. If it's difficult to refine, feel free to discard them since we were liberal with our first step of extracting more targets than less.

### Final Instructions

I've provided some demonstrations below which are some concrete examples that can guide your final output format. Before outputting the final format, think carefully through this task, following the step-by-step instructions outlined above, and provide your reasoning before the final extraction.



---

You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. There are often multiple targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

----

prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the key information of a sentence i.e. the target. For instance, in the sentence, "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Apply this same reductionary thought process that I've demonstrated here to identify the targets. Often times, there are multiple valid targets, but only select the targets such that the sentence can be paragraphased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that the information is true for. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

----


Write each fact as one declarative sentence whose final 1–3 words form the target capturing the key information; the probe is the same sentence with the final target span removed and should read naturally as a cloze. Prefer precise, contentful targets; ensure the last words are exactly the target; avoid cross-references to other facts; preserve the original meaning; extract at most three facts per sentence; and present results in the demonstrated Probe/Target format.

    prompt['system'] = """You will be given two inputs: (1) a full paragraph for context and (2) a single sentence drawn from that paragraph. Your task is to rewrite that sentence into 1–3 self-contained, atomic facts that can stand entirely on their own as probe–target pairs. Use the paragraph only to supply whatever context is needed to make each fact standalone; explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers. Also, do not introduce information not entailed by the sentence. Write each fact as one declarative sentence whose final 1–3 words form the target capturing the key information; the probe is the same sentence with the final target span removed and should read naturally as a cloze. Prefer precise, contentful targets; ensure the last words are exactly the target; avoid cross-references to other facts; preserve the original meaning; extract at most three facts per sentence; and present results in the demonstrated Probe/Target format.

-----
"You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. There are often multiple targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target."

Adjust this prompt above to request the following: "take the output and double check if the probes have properly satisfied the conditions or can be even improved i.e. can be formatted in a more natural way, stays true to the original meaning of the sentence, fully contextualizes the sentence to be self-contained, the target is non-trivial and a central part of the sentence, and is rewritten to naturally place the target at the end. Please filter out probes that that don't meet these conditions and refine the remaining probes."

--

reflection_prompt = """
You are a quality control assistant. Your task is to review and refine the output of a previous process. The original task was to deconstruct a sentence from a paragraph into self-contained, atomic facts.

## Background: The Original Task

The process you are reviewing was as follows:
1.  **Input:** A full paragraph for context and a single sentence from that paragraph.
2.  **Step 1: Identify Targets.** The primary goal was to identify the key, central information of the sentence, called a "target".
    * A valid **target** must be **1-2 words** long.
    * A target must **not** be a mathematical expression (e.g., in LaTeX).
    * Linguistically, a target is often the main predicate (verb), the subjective complement, or even the object of the preposition.
    * Multiple targets may exist for a sentence.
3.  **Step 2: Rewrite into Atomic Probes.** For each valid target, the original sentence was to be rewritten into a self-contained, atomic sentence (a "probe").
    * The probe must be **fully self-contained**, meaning it's completely understandable without the original paragraph. This requires adding context, defining acronyms, and resolving pronouns.
    * Crucially, the probe **must end exactly with the target**.
    * If a sentence could not be rewritten to end with a target, that target was to be discarded.

## Your Task Now: Self-Reflection and Correction

Critically review a provided list of `(target, rewritten_sentence)` pairs against the original rules. For each pair, perform the following checks.

#### Check 1: Target Validation
* **Word Count:** Is the target **exactly one or two words**?
* **Content:** Does the target contain any mathematical notation? (It shouldn't).

If a target fails any of these checks, the entire pair is **invalid**.

#### Check 2: Rewritten Sentence Validation
* **Target Placement:** Does the sentence actually end with the target word(s)? This is the most important rule.
* **Self-Contained:** Is the sentence **fully understandable on its own**? Check for undefined acronyms, unresolved pronouns (like 'it', 'they', 'this'), or ambiguous terms that need clarification from the original context.
* **Clarity:** Does the sentence clearly express **a clear fact**?
* **Accuracy:** Does the rewritten sentence **preserve the exact meaning** of the original without distortion?

If the rewritten sentence fails any of these checks, the entire pair is **invalid**.

### Final Instructions

Based on your review:
1.  **Filter:** Discard any pair that fails the validation checks above. 
2.  **Refine:** For pairs that are valid but could be improved, edit the rewritten sentence.
3.  **Format:** Format the final, filtered, and refined list of targets and their corresponding atomic sentences into JSON with the following keys.

"pairs": list of (target, rewritten probe) 
"""
